# Step 4: Casting parameters

The casting parameters describe the caster's motion: how the rod butt
rotates, and how the hand moves, through the stroke. In the library a
stroke is a `Stroke`: a table of time, rod-butt angle and hand position,
stored as a `stroke.json` file plus a `stroke.csv` table. Time is in
seconds relative to RSP (Rod Straight Position), the angle is the rod
tangent at the handle measured from the casting direction `+x`, and the
hand position is in metres with `+y` up.

**Limits of this step today.**

- A stroke cannot yet be extracted from video or from sensor data. The
  only real stroke bundled is Cast #1, fitted by eye to the footage.
- The simulated cast is sensitive to how a coarsely sampled stroke is
  interpolated, so a coarse stroke gives unreliable results.
  `load_stroke` warns when the time step in a table is above 10 ms.
- A loaded stroke drives `simulate_cast1`. The generic cast
  `simulate_cast` still takes only two stroke parameters: the sweep angle
  and the stroke duration.

The `flycastsim` workflow has five steps. Step 1 stands on its own, and
steps 2 to 4 each produce parameters that step 5 uses.

1. Simple 1-D model (`01_simple_1d_model.ipynb`)
2. Rod characteristics (`02_rod_characteristics.ipynb`)
3. Line and leader characteristics (`03_line_and_leader.ipynb`)
4. Casting parameters (`04_casting_parameters.ipynb`)
5. Simulate a fly cast (`05_simulate_cast.ipynb`)

`complete_workflow.ipynb` runs steps 2 to 5 in sequence.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

from flycastsim.fem import (
    Stroke,
    bundled_stroke_path,
    casting_stroke,
    load_stroke,
    save_stroke,
)

## The idealised stroke

`casting_stroke(sweep, t_stroke)` sweeps the rod butt smoothly through
the angle `sweep` in the time `t_stroke` and then holds it. The angular
velocity is zero at both ends.

In [ ]:
t = np.linspace(0.0, 0.8, 401)
fig = go.Figure()
for sweep_deg, t_stroke in ((120, 0.4), (120, 0.25), (90, 0.4)):
    theta = casting_stroke(np.radians(sweep_deg), t_stroke)
    angle = np.degrees([theta(ti) for ti in t])
    fig.add_trace(go.Scatter(x=t, y=angle,
                             name=f"sweep {sweep_deg}°, {t_stroke} s"))
fig.update_layout(title="Idealised casting stroke",
                  xaxis_title="time [s]",
                  yaxis_title="rod-butt angle from horizontal [deg]")
fig

## The Cast #1 stroke

The Cast #1 stroke from *The Rod & The Cast* (Løvoll & Borger) is bundled
with the package and loads by name. It is sampled every 2.5 ms. The rod
starts up and back and sweeps forward, and the hand moves forward and a
little up until the stop at RSP.

In [ ]:
stroke = load_stroke("cast1")
print(stroke.name)
print(f"{len(stroke.t)} samples from {stroke.t_span[0]} s to {stroke.t_span[1]} s")
print("Files:", bundled_stroke_path("cast1").parent)
print("Method:", stroke.meta["source"]["method"])

t1 = np.linspace(-0.4, 1.0, 701)
angle1 = stroke.angle(t1)

fig = go.Figure(go.Scatter(x=t1, y=np.degrees(angle1), name="Cast #1"))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Cast #1 stroke",
                  xaxis_title="time relative to RSP [s]",
                  yaxis_title="rod-butt angle from horizontal [deg]")
fig

The angular velocity shows where the caster accelerates and stops the
rod.

In [ ]:
omega1 = np.gradient(angle1, t1)
fig = go.Figure(go.Scatter(x=t1, y=np.degrees(omega1)))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Cast #1 angular velocity",
                  xaxis_title="time relative to RSP [s]",
                  yaxis_title="angular velocity [deg/s]")
fig

In [ ]:
peak = np.argmax(np.abs(omega1))
print(f"Total sweep: {np.degrees(angle1.max() - angle1.min()):.0f}°")
print(f"Peak angular velocity: {abs(np.degrees(omega1[peak])):.0f}°/s "
      f"at t = {t1[peak]:+.3f} s")

## The hand path

The hand (the rod butt) also moves. The solver is given the hand
velocity, which is the derivative of the interpolated hand position.

In [ ]:
hand_x, hand_y = stroke.hand_xy(t1)
hand_vx, _ = stroke.hand_vel(t1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t1, y=hand_x, name="x [m]"))
fig.add_trace(go.Scatter(x=t1, y=hand_y, name="y [m]"))
fig.add_trace(go.Scatter(x=t1, y=hand_vx, name="forward speed [m/s]"))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Cast #1 hand path",
                  xaxis_title="time relative to RSP [s]")
fig

## Make, save and load a stroke

A stroke can be built from arrays and written to files with
`save_stroke`. Here the Cast #1 stroke is slowed down by 25 %, keeping
RSP at `t = 0`, then saved and loaded again. Pass the stroke, or the path
to its files, to `simulate_cast1(stroke=...)` (see step 5).

In [ ]:
slow = Stroke(name="Cast #1, 25 % slower", t=stroke.t * 1.25,
              angle_deg=stroke.angle_deg, hand_x=stroke.hand_x,
              hand_y=stroke.hand_y,
              meta={"description": "Cast #1 stroke with times scaled by 1.25"})

with tempfile.TemporaryDirectory() as folder:
    path = save_stroke(slow, Path(folder) / "slow_stroke")
    print("Wrote", sorted(p.name for p in path.parent.iterdir()))
    loaded = load_stroke(path)

print(f"{loaded.name}: {len(loaded.t)} samples, "
      f"step {np.diff(loaded.t).max() * 1000:.3f} ms")

fig = go.Figure()
fig.add_trace(go.Scatter(x=t1, y=np.degrees(stroke.angle(t1)), name="Cast #1"))
fig.add_trace(go.Scatter(x=t1, y=np.degrees(loaded.angle(t1)), name=loaded.name))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(xaxis_title="time relative to RSP [s]",
                  yaxis_title="rod-butt angle from horizontal [deg]")
fig